# 02 — Monthly close
Load billing, compare BEFORE/SOURCE/AFTER states, and replace the month. Archiving stays disabled while DEV and PROD use the same RAW source.

In [ ]:
from pathlib import Path
import sys

source_root = next((root / "src" for root in (Path.cwd(), *Path.cwd().parents) if (root / "src").is_dir()), None)
if source_root is not None and str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))

In [ ]:
ENVIRONMENT = "dev"
MONTH = ""
SOURCE_URI = ""  # empty = derive the path from config/common.toml
try:
    dbutils.widgets.text("environment", ENVIRONMENT)
    dbutils.widgets.text("month", MONTH)
    dbutils.widgets.text("source_uri", SOURCE_URI)
    ENVIRONMENT = dbutils.widgets.get("environment")
    MONTH = dbutils.widgets.get("month")
    SOURCE_URI = dbutils.widgets.get("source_uri")
except NameError:
    pass
if not MONTH:
    raise ValueError("Set month using YYYY-MM.")

In [ ]:
from finops_cloud.pipelines.monthly_close import run

result = run(
    ENVIRONMENT,
    MONTH,
    source_uri=SOURCE_URI or None,
)
display(result)

In [ ]:
from finops_cloud.config import load_config
from finops_cloud.runtime import get_spark

config = load_config(ENVIRONMENT)
spark_session = get_spark(config.profile)
snapshots = spark_session.table(config.table("month_snapshot", "ops"))
display(snapshots.where(f"environment = '{ENVIRONMENT}' AND billing_month = '{MONTH}'").orderBy("captured_at"))